In [1]:
import xarray as xr
import rasterio
from shapely.geometry import Polygon
import geopandas as gpd
import numpy as np
import os
import pandas as pd
from utils.data_prep import h3_grid as h3
from utils.ahp import Ahp_calc
from utils.geo_score_converter import GeoIntervalScorer
%load_ext autoreload
%autoreload 2

In [2]:
# wind_speed
path = '/home/jta/Documentos/articulo_mcdm_lca/data/TEC-001/Velocidad_100m_patched.tif'
ds_wind = rasterio.open(path)
crs = 'EPSG:9377'
# Se corre malla resolucion 6 por temas de computo
resolution = 6

h3_module = h3(
    raster=ds_wind,
    resolution=6,
    crs=crs
)

path = 'seeds/codigo_tipos(in).csv'
df_data = pd.read_csv(path,sep=';',encoding='latin-1')
df_data = df_data.drop(['Criterio'], axis = 1)
# df_data = df_data[~df_data['Subcriterio'].isna()]

## Preparación Malla

In [3]:
path = '/home/jta/Documentos/articulo_mcdm_lca/data'
files = os.listdir(path)
# Mantiene solo archivos que se encuentran en los datos
mask = df_data['Codigo'].isin(files)
df_data = df_data[mask].reset_index(drop=True)

In [4]:
malla_generada =h3_module.vars_ahp(path=path,df_data=df_data)

EPSG:9377
procesando archivo AMB-001-AR
procesando archivo AMB-003-A
procesando archivo AMB-004-A
procesando archivo AMB-005-A
procesando archivo AMB-006-A


/home/jta/miniconda3/envs/cds/lib/python3.14/site-packages/geopandas/geodataframe.py:1969: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


procesando archivo AMB-007-AT
procesando archivo AMB-008-AT


/home/jta/miniconda3/envs/cds/lib/python3.14/site-packages/numpy/_core/_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/home/jta/miniconda3/envs/cds/lib/python3.14/site-packages/numpy/_core/_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/home/jta/miniconda3/envs/cds/lib/python3.14/site-packages/numpy/_core/_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/home/jta/miniconda3/envs/cds/lib/python3.14/site-packages/numpy/_core/_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/home/jta/miniconda3/envs/cds/lib/python3.14/site-packages/numpy/_core/_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/home/jta/

procesando archivo AMB-012-A
procesando archivo AMB-017-A
procesando archivo AMB-028-AT


/home/jta/miniconda3/envs/cds/lib/python3.14/site-packages/geopandas/geodataframe.py:1969: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


procesando archivo AMB-033-A
procesando archivo AMB-040-A
procesando archivo AMB-042-A
procesando archivo TEC-023
procesando archivo TEC-032
procesando archivo TEC-001
procesando archivo TEC-024
procesando archivo SOC-025
procesando archivo SOC-026


In [26]:
# Actualiza las unidades de los datos
malla_generada['TEC-023'] = malla_generada['TEC-023']/1000
malla_generada['TEC-032'] = malla_generada['TEC-032']/1000 

Conversión de los datos a los intervalos. A partir de la malla generada. Los datos de litología se convierten a partir de la información suministrada por Manuela

In [33]:
# Primero convierte los datos de litología
def score_litologia(litol):

    #En caso que halla un valor nan
    if pd.isna(litol):
        score = 0
    else:
        # Lectura de archivo de scores
        path_scores = 'seeds/score_litologia.csv'
        df = pd.read_csv(path_scores)
        # Extrae solo la litologia de interes
        mask = df['Descripcio'] == litol
        df_slice = df[mask]
        # Extra el score asociado
        score = df_slice.iloc[0,1]
    return score

# Actualizacion
malla_generada['AMB-028-AT'] = malla_generada['AMB-028-AT'].apply(score_litologia) 


Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/home/jta/miniconda3/envs/cds/lib/python3.14/site-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_21491/700173494.py", line 19, in <module>
    malla_generada['AMB-028-AT'] = malla_generada['AMB-028-AT'].apply(score_litologia)
                                   ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/home/jta/miniconda3/envs/cds/lib/python3.14/site-packages/pandas/core/series.py", line 4943, in apply
    ).apply()
      ~~~~~^^
  File "/home/jta/miniconda3/envs/cds/lib/python3.14/site-packages/pandas/core/apply.py", line 1422, in apply
    return self.apply_standard()
           ~~~~~~~~~~~~~~~~~~~^^
  File "/home/jta/miniconda3/envs/cds/lib/python3.14/site-packages/pandas/core/apply.py", line 1502, in apply_standard
    mapped = obj._map_values(
        mapper=curried, na_ac

In [39]:
malla = gpd.read_file('malla_articulo')

In [34]:
# Variables a convertir
mask = df_data['Tipo'] == 'excluyente'
df_vars = df_data[~mask]
convertir = list(df_data['Codigo'])

In [40]:
trans = GeoIntervalScorer('seeds/intervalos.csv','seeds/codigo_tipos(in).csv')
temp = trans.transform(malla_generada, columns=convertir)